In [ ]:
"""
File này sẽ thử khả năng dịch của vinai vi2en translate. 
Lý do chọn: state of the art model 2022 và đã được huấn luyện dựa
trên các bệnh án tiếng việt.
"""

In [ ]:
!pip -q install -U transformers sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "vinai/vinai-translate-vi2en-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Required: specify that the source language is Vietnamese
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="vi_VN")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

def translate_vi2en(vi_text):
    inputs = tokenizer(
        vi_text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            # Required: specify that the target language is English
            decoder_start_token_id=tokenizer.lang_code_to_id["en_XX"],
            num_beams=5,
            early_stopping=True,
            max_new_tokens=128
        )

    return tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]

vietnamese_texts = [
"""
1.  Tiền sử bệnh nội khoa
    Các bệnh lý mãn tính: nghi ngờ xơ gan do rượu
    Các yếu tố nguy cơ liên quan: Có tiền sử sử dụng rượu bia

2.  Tiền sử bệnh hiện tại
    Triệu chứng hiện tại: hội chứng não gan
    Các sự kiện trước khi nhập viện
    - đến bệnh viện  với hội chứng não gan
    - Bệnh nhân được chuyển viện trong tình trạng hội chứng não gan tiến triển, ý thức suy giảm
    - Bệnh nhân nhập viện vào ngày w tiến triển hội chứng não gan
    - không thể tự chăm sóc bản thân tại nhà

3.  Đánh giá tại bệnh viện
    Các thủ thuật đã thực hiện: đặt shunt dẫn lưu tĩnh mạch cửa qua da
"""
]

for vi_text in vietnamese_texts:
    print("VI:", vi_text)
    print("EN:", translate_vi2en(vi_text))
    print()

EN: 1. Medical history Chronic disorders: suspected alcoholic cirrhosis Relevant risk factors: history of alcohol use 2. Current medical history Current symptoms: hepatic encephalopathy Events before admission - arrival at the hospital with hepatic encephalopathy - The patient was transferred to the hospital in a state of progressive hepatic encephalopathy, impaired consciousness - The patient was admitted on the day w of progression of hepatic encephalopathy - unable to take care of himself at home 3. Evaluation at the hospital Procedures performed:


### Đánh giá:
Chạy khá tốt đấy.